In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.instacart_silver;

-- Silver: aisles (dimension) - trimmed, deduplicated
CREATE OR REPLACE TABLE instacart_silver.aisles AS
SELECT
  aisle_id,
  TRIM(aisle) AS aisle
FROM instacart.aisles
WHERE aisle_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY aisle_id ORDER BY aisle) = 1;

-- Silver: departments (dimension) - trimmed, deduplicated
CREATE OR REPLACE TABLE instacart_silver.departments AS
SELECT
  department_id,
  TRIM(department) AS department
FROM instacart.departments
WHERE department_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY department) = 1;

-- Silver: products (dimension) - trimmed, deduplicated
-- Note: 1 product (id=6816) has NULL aisle_id/department_id; kept as-is for referential transparency
CREATE OR REPLACE TABLE instacart_silver.products AS
SELECT
  product_id,
  TRIM(product_name) AS product_name,
  aisle_id,
  department_id
FROM instacart.products
WHERE product_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY product_name) = 1;

-- Silver: orders (fact) - validated, type-refined
CREATE OR REPLACE TABLE instacart_silver.orders AS
SELECT
  order_id,
  user_id,
  eval_set,
  order_number,
  order_dow,
  order_hour_of_day,
  CAST(days_since_prior_order AS INT) AS days_since_prior_order
FROM instacart.orders
WHERE order_id IS NOT NULL
  AND user_id IS NOT NULL
  AND eval_set IN ('prior', 'train', 'test')
  AND order_dow BETWEEN 0 AND 6
  AND order_hour_of_day BETWEEN 0 AND 23;

-- Silver: order_products (fact) - FK-validated
CREATE OR REPLACE TABLE instacart_silver.order_products AS
SELECT
  op.order_id,
  op.product_id,
  op.add_to_cart_order,
  op.reordered
FROM instacart.order_products op
LEFT JOIN instacart_silver.orders o ON op.order_id = o.order_id
LEFT JOIN instacart_silver.products p ON op.product_id = p.product_id
WHERE op.order_id IS NOT NULL
  AND op.product_id IS NOT NULL
  AND op.reordered IN (0, 1)
  AND op.add_to_cart_order > 0;

In [0]:
%sql
-- Bronze vs Silver row count comparison
SELECT 'aisles' AS tbl,
  (SELECT COUNT(*) FROM instacart.aisles) AS bronze,
  (SELECT COUNT(*) FROM instacart_silver.aisles) AS silver
UNION ALL
SELECT 'departments',
  (SELECT COUNT(*) FROM instacart.departments),
  (SELECT COUNT(*) FROM instacart_silver.departments)
UNION ALL
SELECT 'products',
  (SELECT COUNT(*) FROM instacart.products),
  (SELECT COUNT(*) FROM instacart_silver.products)
UNION ALL
SELECT 'orders',
  (SELECT COUNT(*) FROM instacart.orders),
  (SELECT COUNT(*) FROM instacart_silver.orders)
UNION ALL
SELECT 'order_products',
  (SELECT COUNT(*) FROM instacart.order_products),
  (SELECT COUNT(*) FROM instacart_silver.order_products)

In [0]:
%sql
-- Validation 1: Primary Key Uniqueness Check
-- Ensures each primary key column has no duplicates

SELECT
  'aisles' AS table_name,
  'aisle_id' AS pk_column,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT aisle_id) AS unique_values,
  COUNT(*) - COUNT(DISTINCT aisle_id) AS duplicate_count,
  CASE 
    WHEN COUNT(*) = COUNT(DISTINCT aisle_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END AS validation_status
FROM instacart.aisles

UNION ALL

SELECT
  'departments',
  'department_id',
  COUNT(*),
  COUNT(DISTINCT department_id),
  COUNT(*) - COUNT(DISTINCT department_id),
  CASE 
    WHEN COUNT(*) = COUNT(DISTINCT department_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.departments

UNION ALL

SELECT
  'products',
  'product_id',
  COUNT(*),
  COUNT(DISTINCT product_id),
  COUNT(*) - COUNT(DISTINCT product_id),
  CASE 
    WHEN COUNT(*) = COUNT(DISTINCT product_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.products

UNION ALL

SELECT
  'orders',
  'order_id',
  COUNT(*),
  COUNT(DISTINCT order_id),
  COUNT(*) - COUNT(DISTINCT order_id),
  CASE 
    WHEN COUNT(*) = COUNT(DISTINCT order_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.orders

ORDER BY table_name;

In [0]:
%sql
-- Validation 2: Null Values Check on Required Columns
-- Ensures critical columns have no NULL values

SELECT
  'aisles' AS table_name,
  'aisle_id' AS column_name,
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(aisle_id) AS null_count,
  CASE 
    WHEN COUNT(*) - COUNT(aisle_id) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END AS validation_status
FROM instacart.aisles

UNION ALL

SELECT 'aisles', 'aisle', COUNT(*), COUNT(*) - COUNT(aisle),
  CASE WHEN COUNT(*) - COUNT(aisle) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.aisles

UNION ALL

SELECT 'departments', 'department_id', COUNT(*), COUNT(*) - COUNT(department_id),
  CASE WHEN COUNT(*) - COUNT(department_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.departments

UNION ALL

SELECT 'departments', 'department', COUNT(*), COUNT(*) - COUNT(department),
  CASE WHEN COUNT(*) - COUNT(department) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.departments

UNION ALL

SELECT 'products', 'product_id', COUNT(*), COUNT(*) - COUNT(product_id),
  CASE WHEN COUNT(*) - COUNT(product_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.products

UNION ALL

SELECT 'products', 'product_name', COUNT(*), COUNT(*) - COUNT(product_name),
  CASE WHEN COUNT(*) - COUNT(product_name) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.products

UNION ALL

SELECT 'products', 'aisle_id', COUNT(*), COUNT(*) - COUNT(aisle_id),
  CASE WHEN COUNT(*) - COUNT(aisle_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.products

UNION ALL

SELECT 'products', 'department_id', COUNT(*), COUNT(*) - COUNT(department_id),
  CASE WHEN COUNT(*) - COUNT(department_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.products

UNION ALL

SELECT 'orders', 'order_id', COUNT(*), COUNT(*) - COUNT(order_id),
  CASE WHEN COUNT(*) - COUNT(order_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.orders

UNION ALL

SELECT 'orders', 'user_id', COUNT(*), COUNT(*) - COUNT(user_id),
  CASE WHEN COUNT(*) - COUNT(user_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.orders

UNION ALL

SELECT 'order_products', 'order_id', COUNT(*), COUNT(*) - COUNT(order_id),
  CASE WHEN COUNT(*) - COUNT(order_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.order_products

UNION ALL

SELECT 'order_products', 'product_id', COUNT(*), COUNT(*) - COUNT(product_id),
  CASE WHEN COUNT(*) - COUNT(product_id) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM instacart.order_products

ORDER BY table_name, column_name;

In [0]:
%sql
-- Validation 3: Full Row Duplication Check
-- Identifies complete duplicate rows across all columns

WITH aisles_dupes AS (
  SELECT aisle_id, aisle, COUNT(*) AS dup_count
  FROM instacart.aisles
  GROUP BY aisle_id, aisle
  HAVING COUNT(*) > 1
),
departments_dupes AS (
  SELECT department_id, department, COUNT(*) AS dup_count
  FROM instacart.departments
  GROUP BY department_id, department
  HAVING COUNT(*) > 1
),
products_dupes AS (
  SELECT product_id, product_name, aisle_id, department_id, COUNT(*) AS dup_count
  FROM instacart.products
  GROUP BY product_id, product_name, aisle_id, department_id
  HAVING COUNT(*) > 1
),
orders_dupes AS (
  SELECT order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order, COUNT(*) AS dup_count
  FROM instacart.orders
  GROUP BY order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order
  HAVING COUNT(*) > 1
),
order_products_dupes AS (
  SELECT order_id, product_id, add_to_cart_order, reordered, COUNT(*) AS dup_count
  FROM instacart.order_products
  GROUP BY order_id, product_id, add_to_cart_order, reordered
  HAVING COUNT(*) > 1
)

SELECT
  'aisles' AS table_name,
  COUNT(*) AS duplicate_row_count,
  CASE WHEN COUNT(*) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END AS validation_status
FROM aisles_dupes

UNION ALL

SELECT 'departments', COUNT(*),
  CASE WHEN COUNT(*) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM departments_dupes

UNION ALL

SELECT 'products', COUNT(*),
  CASE WHEN COUNT(*) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM products_dupes

UNION ALL

SELECT 'orders', COUNT(*),
  CASE WHEN COUNT(*) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM orders_dupes

UNION ALL

SELECT 'order_products', COUNT(*),
  CASE WHEN COUNT(*) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END
FROM order_products_dupes

ORDER BY table_name;

In [0]:
%sql
-- Validation 4: Value Range and Format Validation
-- Checks that values fall within expected ranges

SELECT
  'orders' AS table_name,
  'order_dow' AS column_name,
  'Range 0-6 (days of week)' AS expected_format,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN order_dow NOT BETWEEN 0 AND 6 THEN 1 ELSE 0 END) AS invalid_count,
  CASE 
    WHEN SUM(CASE WHEN order_dow NOT BETWEEN 0 AND 6 THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END AS validation_status
FROM instacart.orders

UNION ALL

SELECT
  'orders',
  'order_hour_of_day',
  'Range 0-23 (hours)',
  COUNT(*),
  SUM(CASE WHEN order_hour_of_day NOT BETWEEN 0 AND 23 THEN 1 ELSE 0 END),
  CASE 
    WHEN SUM(CASE WHEN order_hour_of_day NOT BETWEEN 0 AND 23 THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.orders

UNION ALL

SELECT
  'orders',
  'order_number',
  'Positive integers',
  COUNT(*),
  SUM(CASE WHEN order_number <= 0 THEN 1 ELSE 0 END),
  CASE 
    WHEN SUM(CASE WHEN order_number <= 0 THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.orders

UNION ALL

SELECT
  'orders',
  'days_since_prior_order',
  'Non-negative values',
  COUNT(*),
  SUM(CASE WHEN days_since_prior_order < 0 THEN 1 ELSE 0 END),
  CASE 
    WHEN SUM(CASE WHEN days_since_prior_order < 0 THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.orders

UNION ALL

SELECT
  'orders',
  'eval_set',
  'Values: prior, train, test',
  COUNT(*),
  SUM(CASE WHEN eval_set NOT IN ('prior', 'train', 'test') THEN 1 ELSE 0 END),
  CASE 
    WHEN SUM(CASE WHEN eval_set NOT IN ('prior', 'train', 'test') THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.orders

UNION ALL

SELECT
  'order_products',
  'reordered',
  'Binary: 0 or 1',
  COUNT(*),
  SUM(CASE WHEN reordered NOT IN (0, 1) THEN 1 ELSE 0 END),
  CASE 
    WHEN SUM(CASE WHEN reordered NOT IN (0, 1) THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.order_products

UNION ALL

SELECT
  'order_products',
  'add_to_cart_order',
  'Positive integers',
  COUNT(*),
  SUM(CASE WHEN add_to_cart_order <= 0 THEN 1 ELSE 0 END),
  CASE 
    WHEN SUM(CASE WHEN add_to_cart_order <= 0 THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.order_products

ORDER BY table_name, column_name;

In [0]:
%sql
-- Validation 5: Referential Integrity Check
-- Ensures foreign keys reference existing primary keys

SELECT
  'order_products → orders' AS relationship,
  'order_id' AS foreign_key,
  COUNT(DISTINCT op.order_id) AS fk_distinct_values,
  COUNT(DISTINCT o.order_id) AS matching_pk_values,
  COUNT(DISTINCT op.order_id) - COUNT(DISTINCT o.order_id) AS orphaned_records,
  CASE 
    WHEN COUNT(DISTINCT op.order_id) = COUNT(DISTINCT o.order_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END AS validation_status
FROM instacart.order_products op
LEFT JOIN instacart.orders o ON op.order_id = o.order_id

UNION ALL

SELECT
  'order_products → products',
  'product_id',
  COUNT(DISTINCT op.product_id),
  COUNT(DISTINCT p.product_id),
  COUNT(DISTINCT op.product_id) - COUNT(DISTINCT p.product_id),
  CASE 
    WHEN COUNT(DISTINCT op.product_id) = COUNT(DISTINCT p.product_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.order_products op
LEFT JOIN instacart.products p ON op.product_id = p.product_id

UNION ALL

SELECT
  'products → aisles',
  'aisle_id',
  COUNT(DISTINCT p.aisle_id) - COUNT(DISTINCT CASE WHEN p.aisle_id IS NULL THEN NULL ELSE 1 END),
  COUNT(DISTINCT a.aisle_id),
  (COUNT(DISTINCT p.aisle_id) - COUNT(DISTINCT CASE WHEN p.aisle_id IS NULL THEN NULL ELSE 1 END)) - COUNT(DISTINCT a.aisle_id),
  CASE 
    WHEN (COUNT(DISTINCT p.aisle_id) - COUNT(DISTINCT CASE WHEN p.aisle_id IS NULL THEN NULL ELSE 1 END)) = COUNT(DISTINCT a.aisle_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.products p
LEFT JOIN instacart.aisles a ON p.aisle_id = a.aisle_id

UNION ALL

SELECT
  'products → departments',
  'department_id',
  COUNT(DISTINCT p.department_id) - COUNT(DISTINCT CASE WHEN p.department_id IS NULL THEN NULL ELSE 1 END),
  COUNT(DISTINCT d.department_id),
  (COUNT(DISTINCT p.department_id) - COUNT(DISTINCT CASE WHEN p.department_id IS NULL THEN NULL ELSE 1 END)) - COUNT(DISTINCT d.department_id),
  CASE 
    WHEN (COUNT(DISTINCT p.department_id) - COUNT(DISTINCT CASE WHEN p.department_id IS NULL THEN NULL ELSE 1 END)) = COUNT(DISTINCT d.department_id) THEN '✓ PASS'
    ELSE '✗ FAIL'
  END
FROM instacart.products p
LEFT JOIN instacart.departments d ON p.department_id = d.department_id;